# Training

In [1]:
!pip install -q ml-collections

In [2]:
import jax

In [3]:
import tensorflow as tf
# Set tensorflow to CPU
tf.config.set_visible_devices([], 'TPU')
tf.config.set_visible_devices([], 'GPU')

2026-05-11 04:27:22.970564: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778473643.242933      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778473643.321440      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778473643.859037      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778473643.859087      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778473643.859091      24 computation_placer.cc:177] computation placer alr

In [4]:
import os
import urllib.request
from urllib.error import HTTPError
import ml_collections
import numpy as np
import jax
from jax import numpy as jnp
from jax.sharding import Mesh
from tokenizers import ByteLevelBPETokenizer

main_rng_key = jax.random.key(18)

In [5]:
!rm -rf de_tokenizer_20_000_vocab_size_model
!rm -rf en_tokenizer_20_000_vocab_size_model
!rm -rf log_dir
!rm -f configs.py tokenizer.py data.py fsdp_model.py fsdp_model_utils.py fsdp_training_utils.py


!mkdir de_tokenizer_20_000_vocab_size_model
!mkdir en_tokenizer_20_000_vocab_size_model

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [6]:
base_url = "https://raw.githubusercontent.com/MiguelSteph/transformer-from-scratch/fsdp/"
def download_file_from_github(file_path: str, file_name: str):
    if not os.path.isfile(file_name):
        file_url = base_url + file_path
        print(f"Downloading {file_url}...")
        try:
            urllib.request.urlretrieve(file_url, file_name)
        except HTTPError as e:
            print("Something went wrong. Please try to download the file directly from the GitHub repository:\n", e)

file_paths = [
    'configs/configs.py',
    'data/tokenizer.py',
    'data/data.py',
    'models/fsdp_model.py',
    'models/fsdp_model_utils.py',
    'training/fsdp_training_utils.py',
    'data/en_tokenizer_20_000_vocab_size_model/merges.txt',
    'data/en_tokenizer_20_000_vocab_size_model/vocab.json',
    'data/de_tokenizer_20_000_vocab_size_model/merges.txt',
    'data/de_tokenizer_20_000_vocab_size_model/vocab.json',
]
file_names = [
    'configs.py',
    'tokenizer.py',
    'data.py',
    'fsdp_model.py',
    'fsdp_model_utils.py',
    'fsdp_training_utils.py',
    'en_tokenizer_20_000_vocab_size_model/merges.txt',
    'en_tokenizer_20_000_vocab_size_model/vocab.json',
    'de_tokenizer_20_000_vocab_size_model/merges.txt',
    'de_tokenizer_20_000_vocab_size_model/vocab.json',
]

for file_path, file_name in zip(file_paths, file_names): 
    download_file_from_github(file_path, file_name)

In [7]:
from configs import get_configs
from fsdp_model import create_transformer_module
from fsdp_training_utils import train_and_evaluate, get_dataset_iterator, fsdp_init, generate_random_batch
from data import load_preprocessed_dataset

base_configs = get_configs()
config = ml_collections.ConfigDict(base_configs)
config.data.test_ds_path = '/kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/test.tfrecord'
config.data.validation_ds_path = '/kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/validation.tfrecord'
config.data.train_ds_path = '/kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/train.tfrecord'

# TO REMOVE

config.data.batch_size = 4
config.optimizer.training_epochs = 6
config.optimizer.steps_per_epochs = 20

train_ds = load_preprocessed_dataset(config.data.train_ds_path, config.data.max_seq_len)
train_ds = train_ds.take(1000)
validation_ds = load_preprocessed_dataset(config.data.validation_ds_path, config.data.max_seq_len)
test_ds = load_preprocessed_dataset(config.data.test_ds_path, config.data.max_seq_len)

de_tokenizer = ByteLevelBPETokenizer(config.data.de_tokenizer_model_path + '/vocab.json',
                                     config.data.de_tokenizer_model_path + '/merges.txt')
de_tokenizer.add_special_tokens(list(config.data.special_tokens))
en_tokenizer = ByteLevelBPETokenizer(config.data.en_tokenizer_model_path + '/vocab.json',
                                     config.data.en_tokenizer_model_path + '/merges.txt')
en_tokenizer.add_special_tokens(list(config.data.special_tokens))

enc_pad_id = de_tokenizer.encode('<|pad|>').ids[0]
dec_pad_id = en_tokenizer.encode('<|pad|>').ids[0]

In [8]:
!rm -rf log_dir 
!rm -f log_dir.zip

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [9]:
config

data:
  batch_size: 4
  de_tokenizer_model_path: de_tokenizer_20_000_vocab_size_model
  en_tokenizer_model_path: en_tokenizer_20_000_vocab_size_model
  max_seq_len: 100
  special_tokens:
  - <|pad|>
  - <|startoftext|>
  - <|endoftext|>
  test_ds_path: /kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/test.tfrecord
  train_ds_path: /kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/train.tfrecord
  validation_ds_path: /kaggle/input/datasets/migsena/de-en-separated-20-000-preprocessed-dataset/validation.tfrecord
  vocab_size: 20000
fsdp:
  data_axis: data
  min_weight_size: 256
model:
  d_proj: 64
  dropout: 0.1
  emb_dim: 512
  ff_d_inner_factor: 4
  num_blocks: 6
  num_heads: 8
optimizer:
  base_lr: 0.0001
  steps_per_epochs: 20
  training_epochs: 6
  warmup_epochs: 4
training_output:
  checkpoint_path: log_dir/checkpoints
  metric_path: log_dir/metrics
  trace_path: log_dir/traces

## Training

In [10]:
# Create the mesh
device_array = np.array(jax.devices())
mesh = Mesh(device_array, (config.fsdp.data_axis,))

# Create the model
model = create_transformer_module(config, enc_pad_id, dec_pad_id)

# Create the train state and get the sharding specs
state, state_specs = fsdp_init(
    model,
    mesh,
    config,
    dec_pad_id,
)

# Call Train_and_evaluate
state = train_and_evaluate(model, 
                           mesh,
                           state,
                           state_specs,
                           config,
                           main_rng_key,
                           train_ds,
                           validation_ds,
                           log_dir_prefix=None,
                           dec_pad_id=dec_pad_id)

Epoch 1


  0%|          | 0/20 [00:00<?, ?it/s]

Training:    Loss: 9.882268905639648    Accuracy: 0.011744966730475426
Validation:  Loss: 9.089895248413086    Accuracy: 0.040670450776815414
Epoch 2


  0%|          | 0/20 [00:00<?, ?it/s]

Training:    Loss: 9.052600860595703    Accuracy: 0.04146341234445572
Validation:  Loss: 8.596879959106445    Accuracy: 0.057522002607584
Epoch 3


...

Training:    Loss: 8.374366760253906    Accuracy: 0.0344218909740448
Validation:  Loss: 8.031781196594238    Accuracy: 0.0472039096057415
Epoch 5


  0%|          | 0/20 [00:00<?, ?it/s]

Training:    Loss: 8.169426918029785    Accuracy: 0.044516392052173615
Validation:  Loss: 7.897622108459473    Accuracy: 0.052801862359046936
Epoch 6


  0%|          | 0/20 [00:00<?, ?it/s]

Training:    Loss: 8.017452239990234    Accuracy: 0.04294736683368683
Validation:  Loss: 7.789698123931885    Accuracy: 0.04350203275680542
